# Chấm lại logits của probe Meta cho mean-class recall (CPU, không train)

Mean-class R@5 lấy trung bình trên khoảng 3.8k lớp action, phần lớn là lớp hiếm. Trong khi đó probe được train theo phân bố lệch của tập
train, nên điểm của các lớp hiếm bị kéo xuống. Notebook này thử 2 cách chỉnh sau khi đã có logits, **không cần train và không cần GPU**:

- **logit adjustment:** `z_c − τ·log prior_c`, với prior là tần suất của lớp trong `EPIC_100_train.csv`. Cách này đẩy các lớp hiếm lên.
- **ghép verb–noun:** `action(v,n) = log_softmax(z_a) + β·(log_softmax(z_v)[v] + log_softmax(z_n)[n])`, sau đó áp logit adjustment.

τ và β được chọn bằng **cross-fitting theo người tham gia**: chọn trên một nửa số người, áp dụng cho nửa còn lại, rồi đổi vai. Như vậy
con số báo cáo không được tune trên chính các clip dùng để đo. Cột "tuned-on-all" (tune trên toàn bộ val) chỉ để tham khảo vì nó lạc quan.

**Cài đặt:** Accelerator **None** (CPU), Internet On. *Add Input* **output** của các notebook đã chạy trước:
- `kaggle_eval_ek100_original` (thư mục `logits_orig`, logits trên video gốc, chính là lần ra 32.71): **quan trọng nhất**;
- `kaggle-cv-probe` (thư mục `logits_feats`, giao thức `action_start` trên bản 292p): không bắt buộc.

Notebook tự tìm mọi thư mục logits trong input. Mỗi thư mục mất khoảng 3 phút.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!pip install -q decord
R = "/kaggle/working/repo"
ANN = "/kaggle/working/ek100-ann"
# Clip / class index (same as in the eval runs; no videos needed)
!python $R/scripts/prepare_ek100.py --download_layout --video_root /kaggle/tmp/ek100 \
    --train_csv $ANN/EPIC_100_train.csv --val_csv $ANN/EPIC_100_validation.csv \
    --video_info $ANN/EPIC_100_video_info.csv --out_dir /kaggle/working/work | grep '"val_clips"'


In [ ]:
import glob, os
dirs = sorted({os.path.dirname(os.path.dirname(f)) for f in glob.glob("/kaggle/input/**/shard*/chunk_*.npz", recursive=True)})
dirs = [d for d in dirs if os.path.basename(d) in ("official", "action_start")]  # not the __oracle / __enconly variants
assert dirs, "no logits found: Add Input the output of kaggle_eval_ek100_original (logits_orig) and/or kaggle-cv-probe"
for d in dirs:
    print(d)


In [ ]:
for i, d in enumerate(dirs):
    name = d.split("/kaggle/input/")[-1].replace("/", "_")
    print(f"\n========== {d} ==========", flush=True)
    !python $R/scripts/posthoc_ek100.py --work_dir /kaggle/working/work --train_csv $ANN/EPIC_100_train.csv \
        --logits_dir {d} --out /kaggle/working/posthoc_{name}.json


Gửi lại output của cell cuối, gồm các bảng `released / cross-fitted` cho từng thư mục logits.